# SNAPGASP Studio — Google Colab

Notebook-native AI image/video workspace. No external GPU server or public tunnel is started.

Run the cells in order. The source code is pulled directly from the project repository.

In [ ]:
from pathlib import Path
import subprocess, sys
repo_url = 'https://github.com/sarada2000sd/snapgasp-studio.git'
project_dir = Path('/content/snapgasp-studio')
if project_dir.exists():
    subprocess.run(['git','-C',str(project_dir),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth','1',repo_url,str(project_dir)],check=True)
sys.path.insert(0,str(project_dir))
print('SNAPGASP source ready:',project_dir)

## 1. Install dependencies
This requirements file intentionally does not reinstall PyTorch, preserving Colab's CUDA-compatible build.

In [ ]:
%pip -q install -r /content/snapgasp-studio/requirements-colab.txt
print('Dependencies installed. If Colab asks for a runtime restart, restart and rerun from the first cell.')

## 2. Check the GPU
Choose Runtime → Change runtime type → T4 GPU (when available). A free GPU is not guaranteed.

In [ ]:
import torch
if torch.cuda.is_available():
    p=torch.cuda.get_device_properties(0)
    free,total=torch.cuda.mem_get_info(0)
    print('GPU:',p.name)
    print('VRAM total:',round(total/2**30,1),'GB')
    print('VRAM free:',round(free/2**30,1),'GB')
else:
    print('No CUDA GPU detected. Select a GPU runtime if Colab offers one; generation will not run without a supported GPU.')

## 3. Mount Google Drive
Model weights, generated media, and history are stored in Drive.

In [ ]:
from google.colab import drive
import os
from pathlib import Path
drive.mount('/content/drive')
os.environ['SNAPGASP_DATA_DIR']='/content/drive/MyDrive/SNAPGASP-Studio'
Path(os.environ['SNAPGASP_DATA_DIR']).mkdir(parents=True,exist_ok=True)
print('Storage:',os.environ['SNAPGASP_DATA_DIR'])

## 4. Launch SNAPGASP Studio
Open Model Manager, install a model that fits the detected GPU, load it, then generate. Only one model stays active at a time.

In [ ]:
from studio.ui import build_dashboard
dashboard=build_dashboard()
display(dashboard)

## Notes
- Colab Free does not guarantee GPU availability, GPU type, VRAM, quota, or session duration.
- Start with the smallest implemented model that fits your available GPU. Large models can run out of memory.
- Gated models may require accepting the model license and a Hugging Face token. Never put tokens in GitHub.
- Diffusers image and CogVideoX 2B text-to-video paths are implemented. Wan 2.2 and HunyuanVideo are planned, not enabled.
- Review each upstream model's license before commercial use.